# Step 1: Import Relevant LIbraries

In [69]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score


## STEP 2: Load the cleaned dataset

In [70]:

cleaned_df = pd.read_csv("/home/nyaugenya/dev/personal/AdvanceDataScience/Water Pump Functionality Prediction/data/cleaned_water_pump_data.csv")
cleaned_df.head()

,amount_tsh,gps_height,region,basin,region_code,construction_year,date_recorded,district_code,population,public_meeting,...,extraction_type_class,management_group,payment_type,water_quality,quantity_group,source_type,waterpoint_type_group,status_group,month_recorded,age
0,6000.0,1390,Iringa,Lake Nyasa,11,1970-01-01 00:00:00.000001999,2011-03-14,5,109,True,...,gravity,user-group,annually,soft,enough,spring,communal standpipe,functional,3,41
1,25.0,686,Manyara,Pangani,21,1970-01-01 00:00:00.000002009,2013-02-25,4,250,True,...,gravity,user-group,per bucket,soft,enough,dam,communal standpipe,functional,2,43
2,0.0,263,Mtwara,Ruvuma/Southern Coast,90,1970-01-01 00:00:00.000001986,2013-01-28,63,58,True,...,submersible,user-group,never pay,soft,dry,borehole,communal standpipe,non functional,1,43
3,20.0,0,Tanga,Pangani,4,1970-01-01 00:00:00.000002009,2011-03-13,8,1,True,...,submersible,user-group,per bucket,salty,enough,other,communal standpipe,functional,3,41
4,0.0,0,Shinyanga,Internal,17,1970-01-01 00:00:00.000000000,2012-10-01,3,0,True,...,handpump,user-group,never pay,soft,enough,borehole,hand pump,non functional,10,42


## STEP 3: SPlit the features and the targets, then split into training and test sets


### SPLIT the target from the features

In [71]:
X = cleaned_df.drop(columns=['status_group']) # write your variable as capital letter if is a 2D array 
y = cleaned_df['status_group'] # and a small letter if is a 1D array

X.shape

(49840, 21)

In [72]:
X.shape

(49840, 21)

In [73]:
y.shape

(49840,)

# split into train and test

In [74]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y,random_state=42) # stratify=y ensures that the class distribution is preserved(maintained) in both the training and testing sets

In [75]:
((cleaned_df['status_group'].value_counts()) / len(cleaned_df)) * 100

status_group
functional                 54.935795
non functional             37.702648
functional needs repair     7.361557
Name: count, dtype: float64

In [76]:
# Encode the target column coz many machine learning algorithms require the target variable to be in numerical format. We can use label encoding to convert the categorical labels into numerical values.
# _ common_types_of_encoders_
# Ordinal Encoder: This encoder is used when the categorical variable has an inherent order or ranking. It assigns a unique integer to each category based on their order. For example, if we have a feature "Education Level" with categories "High School," "Bachelor's," and "Master's," we can assign 0 to "High School," 1 to "Bachelor's," and 2 to "Master's."
# One-Hot Encoder: This encoder is used when the categorical variable does not have any inherent order. It creates binary columns for each category, indicating the presence or absence of that category. For example, if we have a feature "Color" with categories "Red," "Green," and "Blue," one-hot encoding would create three binary columns: "Color_Red," "Color_Green," and "Color_Blue," where each column would have a value of 1 if the corresponding color is present and 0 otherwise.
# Label Encoder: This encoder is used when the categorical variable is the target variable (the variable we want to predict). It assigns a unique integer to each category. For example, if we have a target variable "Animal" with categories "Dog," "Cat," and "Bird," label encoding would assign 0 to "Dog," 1 to "Cat," and 2 to "Bird." This is useful for algorithms that require numerical labels for classification tasks
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.fit(y_test)

## STEP 4: Building the preprocessing pipeline
# Why use sklearn's pipeline?
Using sklearn's pipeline is beneficial for several reasons:
- it bundles the data preprocessing and model training into a single ,cohesive object making your machine learning workflows cleaner,safer and production ready
- It helps to streamline the preprocessing and modeling steps, making the code more readable and maintainable.
- It ensures that the same preprocessing steps are applied to both the training and testing data, reducing the risk of data leakage.
- It allows for easy hyperparameter tuning and cross-validation, as the entire pipeline can be treated as a single estimator.
- Instead of manually tracking and multiple `,fit()` and `.transform()` methods across your training and tsting sets, apipeline allows you to execute the entire sequence with a single command

In [77]:
# Numerical Columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('month_recorded') # remove month_recorded from numerical_cols list since it is a categorical variable
print("Numerical Columns:", numerical_cols)

# Categorical Columns
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
categorical_cols.append('month_recorded') # add month_recorded to categorical_cols list since it is a categorical variable
print("Categorical Columns:", categorical_cols)



Numerical Columns: ['amount_tsh', 'gps_height', 'region_code', 'district_code', 'population', 'age']
Categorical Columns: ['region', 'basin', 'construction_year', 'date_recorded', 'scheme_management', 'extraction_type_class', 'management_group', 'payment_type', 'water_quality', 'quantity_group', 'source_type', 'waterpoint_type_group', 'month_recorded']


/tmp/ipykernel_589702/576661846.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()


### 4.1: NUMERICAL PIPELINE

In [78]:
### 4.1 Numerical Pipeline
# Create a numerical pipeline that imputes missing values with the median and scales the features using StandardScaler. The pipeline consists of two steps: 'imputer' and 'scaler'. The 'imputer' step uses SimpleImputer to fill in missing values with the median of each feature, and the 'scaler' step uses StandardScaler to standardize the features by removing the mean and scaling to unit variance. This pipeline is applied to the numerical columns of the dataset.
numerical_pipeline = Pipeline(steps=[
    # ('imputer', SimpleImputer(strategy='median')), advised to clean your data before training the model
    ('scaler', StandardScaler())
])

### 4.2 CATEGORICAL PIPELINE

In [ ]:
### 4.`2 Categorical Pipeline
# Create a categorical pipeline that imputes missing values with the most frequent value and applies one-hot encoding to the categorical features. The pipeline consists of two steps: 'imputer' and 'onehot'. The 'imputer' step uses SimpleImputer to fill in missing values with the most frequent value of each feature, and the 'onehot' step uses OneHotEncoder to convert categorical features into binary columns. The handle_unknown='ignore' parameter in OneHotEncoder ensures that any unknown categories encountered during transformation are ignored, preventing errors during model training and prediction. This pipeline is applied to the categorical columns of the dataset.`

#handle_unknown='ignore' parameter in OneHotEncoder ensures that any unknown categories encountered during transformation are ignored, preventing errors during model training and prediction. This pipeline is applied to the categorical columns of the dataset.
categorical_pipeline = Pipeline(steps=[
    # ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

### Combine the numerical and categorical pipelines using ColumnTransformer

In [80]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)])